# EV Curves with Quantum ESPRESSO

**Prerequisites**
- `pw.x` must be available on `$PATH` (or pass the full path via `qe_command`)
- A UPF pseudopotential file for the element of interest
- A QE input template file with `&CONTROL`, `&SYSTEM`, `&ELECTRONS` namelists and `K_POINTS`
  (atomic positions and cell will be filled in automatically by ASE)

In [ ]:
import matplotlib.pyplot as plt
from conceptual_dictionary import ConceptualDict
from semantic_workflows.build import bulk
from semantic_workflows.evcurves import relax_structure, calculate_ev_curves

## Setup

Edit the paths and parameters below to match your system.

In [ ]:
# Path to the QE input template (contains namelists + K_POINTS, no atomic structure)
input_template = "qe_ref/pw.si.scf.ref"

# Pseudopotential files: {element_symbol: path_to_UPF}
pseudopotentials = {"Si": "qe_ref/Si.pbe-n-rrkjus_psl.1.0.0.UPF"}

# QE executable (full path if not on $PATH)
qe_command = "pw.x"

## Run the workflow

In [ ]:
cd = ConceptualDict()

# 1. Build initial bulk structure
structure = bulk("Si", crystalstructure="diamond", a=5.43, cubic=True, cdict=cd)

In [ ]:
# 2. Relax with QE vc-relax (ions + cell)
relaxed_structure, ecoh, vol = relax_structure(
    structure,
    engine="qe",
    input_template=input_template,
    pseudopotentials=pseudopotentials,
    working_dir="qe_relax_run",
    qe_command=qe_command,
    cdict=cd,
)
print(f"Relaxed: E = {ecoh:.4f} eV/atom, V = {vol:.4f} Å³/atom")

In [ ]:
# 3. Compute EV curve (SCF at each scaled volume)
result = calculate_ev_curves(
    relaxed_structure,
    engine="qe",
    vol_range=0.1,
    num_of_points=7,
    input_template=input_template,
    pseudopotentials=pseudopotentials,
    working_dir="qe_ev_run",
    qe_command=qe_command,
    cdict=cd,
)
print(f"Bulk modulus = {result['bulk_modulus']:.1f} GPa")

In [ ]:
# 4. Export semantic metadata
cd.to_yaml("qe_evcurve.yaml")

## Plot

In [ ]:
fig, ax = plt.subplots()
ax.plot(result["volume"], result["energy"], label="Birch-Murnaghan fit")
ax.set_xlabel("Volume per atom (Å³)")
ax.set_ylabel("Energy per atom (eV)")
ax.set_title(f"EV curve — Si (QE/PBE)  |  B = {result['bulk_modulus']:.1f} GPa")
ax.legend()
plt.tight_layout()
plt.show()